In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.wkt import loads
from IPython.display import Image
import folium
import matplotlib.pyplot as plt

# Growth Rate Analysis

In [ ]:
Image('../plots/preliminary_linear_analysis.png')

Predicted Growth Rate for each SA2 district between 2021 and 2026

In [ ]:
growth_rates = pd.read_csv("../data/curated/growth_rates.csv")

growth_rates.head()

Predicted Census data for SA2 districts

In [ ]:
sa2_info_2026 = pd.read_csv('../data/curated/suburb_info_2026.csv')

sa2_info_2026.head()

These SA2 codes were turned to postcode granularity by checking each property's SA2 and aligning it with Postcode

In [ ]:
postcode_info_2026 = pd.read_csv("../data/curated/postcode_info_2026.csv")

postcode_info_2026.head()

This census data for each postcode was then fed into the created XGBoost model for the predictions of rental prices in 2026

In [ ]:
predictions = pd.read_csv('../data/curated/predictions_2026.csv')

predictions.head()

We can then compare the growth rates between the predicted 2023 and 2026 data 

In [ ]:
rent_data = pd.read_csv("../data/curated/weekly_rent_2023_and_2026.csv")

rent_data.head()

Map with the top 10 suburbs projected with the highest growth rate

In [ ]:
sf = gpd.read_file("../data/landing/postcode/POA_2021_AUST_GDA2020.shp")
vic_suburbs = sf[sf["POA_CODE21"].astype(str).str.startswith('3')]
geoJSON = vic_suburbs[['POA_CODE21', 'geometry']].drop_duplicates('POA_CODE21').to_json()

vic_suburbs["POA_CODE21"] = vic_suburbs["POA_CODE21"].astype(int)
rent_data["postcode"] = rent_data["postcode"].astype(int)

gdf = gpd.GeoDataFrame(
    rent_data.merge(vic_suburbs, right_on='POA_CODE21', left_on='postcode', how='left')
)

gdf['geometry'] = gdf['geometry'].to_crs("+proj=longlat +ellps=WGS84 +datum=WGS84 +no_defs")
gdf['centroid'] = gdf['geometry'].apply(lambda x: (x.centroid.y, x.centroid.x))

df = gdf[["POA_CODE21", 'growth_rate']]

m = folium.Map(location=[-37.020100, 144.964600], tiles="Stamen Terrain", zoom_start=7)


# refer to the folium documentations on more information on how to plot aggregated data.
c = folium.Choropleth(
    geo_data=geoJSON, # geoJSON 
    name='choropleth', # name of plot
    data=df.reset_index(), # data source
    columns=["POA_CODE21", 'growth_rate'], # the columns required
    key_on= 'properties.POA_CODE21', # this is from the geoJSON's properties
    fill_color='viridis', # color scheme
    bins=10,
    nan_fill_color='white',
    legend_name='Growth rate of Weekly Rent'
)

c.add_to(m)

top_10_suburbs = df.sort_values(by = "growth_rate", ascending=False)["POA_CODE21"][:10].values

for suburb in top_10_suburbs:
    coord = gdf.loc[gdf["postcode"] == suburb]["centroid"].values
    m.add_child(
        folium.Marker(location=coord)
    )
    
m